# 05 · Access scores per block group

Same code as `scoring.run` (the last part of `make score`). Each residential building gets its
network distance to the nearest service of each type, turned into a band value: ≤400 m = 1.0,
≤800 m = 0.6, ≤1600 m = 0.3, else 0. Block-group `<s>_score` is the population-weighted mean ×
100. `access_score` = Σ weightₛ × `<s>_score`. Scores are absolute (not rescaled);
`access_pctile` is the city-relative rank. Convenience stores are a side metric, not in the composite.

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
print(cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year)
print("schema:", cfg.slug, "| EPSG", cfg.analysis_epsg)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from urban_access import db, scoring

eng = db.engine()


def q(sql: str, **params) -> pd.DataFrame:
    with eng.connect() as con:
        return pd.read_sql(text(sql.replace("__S__", f'"{cfg.slug}"')), con, params=params)


def qgeo(sql: str) -> gpd.GeoDataFrame:
    with eng.connect() as con:
        return gpd.read_postgis(text(sql.replace("__S__", f'"{cfg.slug}"')), con, geom_col="geom")

In [ ]:
out = scoring.run(cfg)
svc = cfg.scoring.scored_services
print(len(out), "block groups →", cfg.processed_dir)
out[
    ["geoid", "pop_total", "access_score", "access_rank"] + [f"{s}_score" for s in svc]
].describe().round(1)

## Sanity checks

In [ ]:
assert abs(sum(cfg.scoring.weights.values()) - 1) < 1e-9
assert out["geoid"].is_unique and len(out) == len(q("SELECT geoid FROM __S__.block_groups"))
pop = out["pop_total"] > 0
assert out.loc[pop, "access_score"].between(0, 100).all()
assert out.loc[~pop, "access_score"].isna().all()
recon = sum(cfg.scoring.weights[s] * out[f"{s}_score"] for s in svc)
assert ((recon - out["access_score"]).abs()[pop] < 1e-9).all()
print("zero-population block groups:", out.loc[~pop, "geoid"].tolist())
out[["circuity_ratio", "snap_flag_pop_pct", "n_origins"]].describe().round(2)

In [ ]:
# city-wide share of residents within a 5-minute walk (400 m), per service
od = q("""SELECT od.service, od.dist_m, o.pop FROM __S__.origin_dist od
            JOIN __S__.origins o ON o.id = od.origin_id WHERE o.kind <> 'centroid'""")
od["near"] = od["dist_m"] <= cfg.scoring.near_m
city = od.groupby("service").apply(
    lambda x: pd.Series(
        {
            "residents_5min": x.loc[x["near"], "pop"].sum(),
            "pct_5min": 100 * x.loc[x["near"], "pop"].sum() / x["pop"].sum(),
        }
    ),
    include_groups=False,
)
city.round(1)

## Maps

In [ ]:
classes = [0, 20, 40, 60, 80, 100]  # fixed classes: scores are absolute
fig, axes = plt.subplots(2, 3, figsize=(16, 13))
for ax, col in zip(axes.flat, ["access_score"] + [f"{s}_score" for s in svc], strict=True):
    out.plot(
        ax=ax,
        column=col,
        cmap="viridis",
        scheme="UserDefined",
        classification_kwds={"bins": classes[1:]},
        legend=True,
        edgecolor="white",
        linewidth=0.3,
        missing_kwds={"color": "lightgray", "hatch": "//", "label": "no pop"},
        legend_kwds={"loc": "lower left", "fontsize": 7},
    )
    ax.set_title(col)
    ax.set_axis_off()
plt.tight_layout()

In [ ]:
cols = (
    ["geoid", "pop_total", "access_score"]
    + [f"{s}_score" for s in svc]
    + ["grocery_pop_5min_pct", "people_per_grocery", "median_hh_income", "zero_car_hh_pct"]
)
ranked = out[pop].sort_values("access_score", ascending=False)[cols].round(1)
print("top 5")
print(ranked.head(5).to_string(index=False))
print("bottom 5")
print(ranked.tail(5).to_string(index=False))

In [ ]:
# block groups to spot-check by hand in QGIS (top, median, bottom): open <slug>.bg_scores,
# <slug>.edges, <slug>.dests and <slug>.origins on localhost:5436
spot = pd.concat([ranked.head(1), ranked.iloc[[len(ranked) // 2]], ranked.tail(1)])
spot[["geoid", "access_score", "grocery_score", "transit_score"]]